In [2]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt



In [5]:
# Load frozen Phase 1 input
adata_full = sc.read_h5ad(
    "../data/processed/Filbin_cohort_raw.h5ad"
    )
print(f"Loaded: {adata_full.shape[0]} cells x {adata_full.shape[1]} genes")
print(f"X dtype: {adata_full.X.dtype}")
print(f"Layers: {list(adata_full.layers.keys())}")

Loaded: 4058 cells x 23686 genes
X dtype: float64
Layers: ['tpm']


In [6]:
# Subset to Phase 1 cohort (paper-exact 2,458 cells)
adata = adata_full[adata_full.obs["in_paper_cohort"]].copy()
del adata_full  # free memory

In [7]:
# Cast X and the tpm layer to float32
adata.X = adata.X.astype(np.float32)
adata.layers["tpm"] = adata.layers["tpm"].astype(np.float32)

In [9]:
# Anchor verification
n_cells, n_genes = adata.shape
print(f"\nFilbin cohort: {n_cells} cells x {n_genes} genes")
print(f"Expected: 2458 cells x 23686 genes")
print(f"Anchor match: {n_cells == 2458 and n_genes == 23686}")


Filbin cohort: 2458 cells x 23686 genes
Expected: 2458 cells x 23686 genes
Anchor match: True


In [ ]:
# Per-tumor cell counts (sanity check)
print(f"\nCells per tumor:")
print(adata.obs["patient"].value_counts().sort_index())


Cells per tumor:
BCH836     527
BCH869     492
BCH1126    299
MUV1       146
MUV5       708
MUV10      286
Name: patient, dtype: int64


In [24]:
# Confirm X is still raw TPM at this point (per-cell sum should be ~1e6)
per_cell_sums = np.asarray(adata.X.sum(axis=1)).flatten()
print(f"\nMedian per-cell TPM sum: {np.median(per_cell_sums):.1f}  (expect 1,000,000)")


Median per-cell TPM sum: 1000000.0  (expect 1,000,000)


In [25]:
# --- Substep 3c: compute Ea from raw TPM, identify genes to keep ---
# Ea_i = log2(mean(TPM_i,:) + 1), per Filbin SM.
# Computed on the RAW TPM layer, not on log-transformed data.
tpm = adata.layers["tpm"]                       # (cells, genes), float32
mean_tpm_per_gene = np.asarray(tpm.mean(axis=0)).flatten()
Ea = np.log2(mean_tpm_per_gene + 1.0)

# Store Ea in var so it persists and can be re-inspected
adata.var["Ea"] = Ea

# Filter: drop genes with Ea < 4
keep_genes = Ea >= 4
n_keep = int(keep_genes.sum())
n_drop = int((~keep_genes).sum())
print(f"Genes retained (Ea >= 4): {n_keep}")
print(f"Genes dropped  (Ea <  4): {n_drop}")
print(f"Total:                    {n_keep + n_drop}  (expect 23686)")

# --- Substep 3b: apply Filbin transform to .X (still on full gene set) ---
# E = log2(TPM/10 + 1). /10 rationale: TPM normalizes to 1e6 transcripts,
# but single cells contain ~1e5 transcripts; /10 aligns the +1 pseudocount
# with the actual single-cell detection limit. Suva-lab convention.
adata.X = np.log2(adata.layers["tpm"] / 10.0 + 1.0).astype(np.float32)

# Quick sanity: E values should be non-negative, finite, modest range
X_arr = adata.X
print(f"\nE matrix dtype: {X_arr.dtype}")
print(f"E min/max:      {X_arr.min():.3f} / {X_arr.max():.3f}")
print(f"E mean (overall): {X_arr.mean():.3f}")
print(f"Fraction of E values == 0: {(X_arr == 0).mean():.3f}  (expect ~0.76 — matches raw zero fraction)")

# --- Now apply the gene filter to both .X and layers["tpm"] ---
adata = adata[:, keep_genes].copy()
print(f"\nAfter gene filter: {adata.shape[0]} cells x {adata.shape[1]} genes")

Genes retained (Ea >= 4): 8494
Genes dropped  (Ea <  4): 15192
Total:                    23686  (expect 23686)

E matrix dtype: float32
E min/max:      0.000 / 15.150
E mean (overall): 0.595
Fraction of E values == 0: 0.775  (expect ~0.76 — matches raw zero fraction)

After gene filter: 2458 cells x 8494 genes
